# The helicity barrier in FLR-MHD

Companion of `examples/helicity_barrier_run.py` (plans/FLRMHD_PLAN.md rung F7): the three runs of Meyrand, Squire, Schekochihin & Dorland (2021, JPP 87, 535870301) Fig. 3 at the same energy injection rate $\varepsilon$ —

| run | model | forcing |
|---|---|---|
| `imb` | FLR-MHD | imbalanced, $\sigma_\varepsilon=\varepsilon_H/\varepsilon=0.88$ |
| `bal` | FLR-MHD | balanced, $\sigma_\varepsilon=0$ |
| `rmhd` | $\rho_i=0$ (RMHD) | imbalanced, $\sigma_\varepsilon=0.88$ |

**Model** (Gaussian units; derivation in docs/numerics.md "FLR-MHD"). With $\mathbf u_\perp=(c/B_0)\hat{\mathbf z}\times\nabla_\perp\varphi$, $\mathbf b_\perp=-(1/B_0)\hat{\mathbf z}\times\nabla_\perp A_\parallel$, $\Omega_i=ZeB_0/(m_ic)$, $\rho_i=c\sqrt{2m_iT_{0i}}/(ZeB_0)$ and $v_A=B_0/\sqrt{4\pi n_{0i}m_i}$,

$$\Big(\frac{\partial}{\partial t}+\mathbf u_\perp\!\cdot\!\nabla_\perp\Big)\frac{\delta n_e}{n_{0e}}=-\frac{c}{4\pi e n_{0e}}\Big(\frac{\partial}{\partial z}+\mathbf b_\perp\!\cdot\!\nabla_\perp\Big)\nabla_\perp^2A_\parallel,\qquad
\Big(\frac{\partial}{\partial t}+\mathbf u_\perp\!\cdot\!\nabla_\perp\Big)A_\parallel=-c\frac{\partial\varphi}{\partial z}+\frac{cT_{0e}}{e}\Big(\frac{\partial}{\partial z}+\mathbf b_\perp\!\cdot\!\nabla_\perp\Big)\frac{\delta n_e}{n_{0e}},$$

$$\frac{\delta n_e}{n_{0e}}=-\frac{Z}{\tau}\big(1-\hat\Gamma_0\big)\frac{e\varphi}{T_{0e}},\qquad \tau=T_{0i}/T_{0e}.$$

Linear waves have $\omega=\pm k_zv_Av_{\rm ph}(k_\perp)$ with $v_{\rm ph}\to1$ for $k_\perp\rho_i\ll1$ (Alfvén) and $\propto k_\perp\rho_i$ for $k_\perp\rho_i\gg1$ (kinetic Alfvén). The invariants are the energy $E=E^++E^-$ and the generalized helicity $H=\sum_k(E^+_k-E^-_k)/v_{\rm ph}(k_\perp)$, with $E^\pm=\tfrac14\langle|\nabla_\perp\Theta^\pm|^2\rangle$ the generalized Elsasser energies. Because $v_{\rm ph}$ grows with $k_\perp$, helicity injected at large scales cannot follow the energy to small scales: the **helicity barrier**. The run is in code units, $v_A=1$, cubic box $L=2\pi$ (so $k_{\min}=2\pi/L=1$), $\tau_A=L/v_A$.

**Pass criteria (plan §5 F7)** evaluated below from whatever output exists:

1. controls saturate: `bal` and `rmhd` reach $\varepsilon_\perp+\varepsilon_z=\varepsilon$ within 15% (time-averaged), with $\varepsilon_\perp\gg\varepsilon_z$ for `bal`;
2. `imb` does not: $E$ grows, the budget $dE/dt=\varepsilon-\varepsilon_\perp-\varepsilon_z$ closes to 5%, and $\varepsilon_\perp/\varepsilon$ falls to within a factor 2 of $1-\sigma_\varepsilon$ while `rmhd` keeps $\varepsilon_\perp/\varepsilon>0.5$;
3. flux blocked: time-averaged $\Pi(k)/\varepsilon<0.3$ for $k>2k_*$ in `imb`, $\Pi/\varepsilon\approx1$ in `bal`;
4. $E^+(k)$ breaks at $k_*\rho_i<1$ to a slope steeper than $-3$; $k_*\rho_i$ follows $(1-\sigma_c)^{1/4}$ within a factor 2;
5. (stretch) $E_{\rm sat}$ rises as $\nu_z$ falls — needs a $\nu_z$ scan, not produced by the default script invocation.

Nothing here is hardcoded: every number is computed from the files found. **The `smoke` profile ($24^3$, $2\,\tau_A$) only proves the machinery — its numbers are not physics**, and the verdict cells say so.

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt

import helicity_barrier_run as hb
from taranis.diagnostics import flrmhd as dgf

ROOT = os.environ.get("TARANIS_HB_ROOT", hb.DEFAULT_ROOT)
# profile: the one asked for, else the most ambitious one with any output on disk
PROFILE = os.environ.get("TARANIS_HB_PROFILE") or next(
    (p for p in ("full", "m1", "smoke")
     if any(os.path.exists(os.path.join(ROOT, p, r, "timeseries.txt")) for r in hb.RUNS)), "smoke")

# three runs = three categorical slots, in fixed order (colour follows the run everywhere)
COLOR = {"imb": "#2a78d6", "bal": "#eb6834", "rmhd": "#1baf7a"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "lines.linewidth": 2.0,
                     "legend.frameon": False, "font.size": 10})

runs = {}
for r in hb.RUNS:
    d = os.path.join(ROOT, PROFILE, r)
    ts = hb.load_timeseries(d)
    if ts is None:
        continue
    with open(os.path.join(d, "meta.json")) as f:
        meta = json.load(f)
    runs[r] = dict(ts=ts, spec=hb.load_spectra(d), meta=meta)

print(f"profile {PROFILE!r} under {ROOT}")
if not runs:
    print("no output found -- run e.g.  TARANIS_HB_PROFILE=smoke python examples/helicity_barrier_run.py")
for r, v in runs.items():
    m, ts = v["meta"], v["ts"]
    print(f"  {r:5s} {hb.RUN_LABELS[r]:30s} {m['n']}^3  k_min rho_i = {m['rho_i_run']:g}  sigma_eps = "
          f"{m['sigma_eps_run']:g}  t = {ts['t'][-1]/m['tau_A']:.2f} of {m['t_end']/m['tau_A']:g} tau_A  "
          f"({len(ts['t'])} rows, {len(v['spec'])} spectra, {int(ts['step'][-1])} steps)")
SMOKE = PROFILE == "smoke"


def window(v, frac=0.5):
    # the analysis window: the last `frac` of the time covered so far
    t = v["ts"]["t"]
    return t >= t[-1] - frac*(t[-1] - t[0])


def tmean(v, y, frac=0.5):
    # time average of a per-block series over the window (rows are block ends)
    ts = v["ts"]; w = window(v, frac) & np.isfinite(ts["dt"]) & np.isfinite(y)
    span = np.diff(ts["t"], prepend=ts["t"][0])
    return np.sum(y[w]*span[w])/np.sum(span[w])


def spec_window(v, frac=0.5):
    sp = v["spec"]
    if not sp:
        return []
    t = np.array([float(s["t"]) for s in sp])
    keep = t >= t[-1] - frac*(t[-1] - t[0])
    # the main phase's first snapshot repeats the spin-up's last time: keep one per time
    out, seen = [], set()
    for s, k in zip(sp, keep):
        if k and float(s["t"]) not in seen:
            seen.add(float(s["t"])); out.append(s)
    return out

## Energy and dissipation histories (criteria 1 and 2)

Energies in units of $\varepsilon\tau_A$, rates in units of $\varepsilon$. The barrier's signature is in the right-hand column: perpendicular dissipation falling toward $(1-\sigma_\varepsilon)\varepsilon$ while the energy keeps growing, until parallel dissipation $\varepsilon_z$ takes over.

In [ ]:
if runs:
    fig, ax = plt.subplots(2, 2, figsize=(11, 6.5), sharex=True)
    for r, v in runs.items():
        ts, m = v["ts"], v["meta"]
        t, u = ts["t"]/m["tau_A"], m["eps"]*m["tau_A"]
        ax[0, 0].plot(t, ts["E"]/u, color=COLOR[r], label=hb.RUN_LABELS[r])
        ax[1, 0].plot(t, ts["E_plus"]/u, color=COLOR[r])
        ax[1, 0].plot(t, ts["E_minus"]/u, color=COLOR[r], ls="--")
        ax[0, 1].plot(t, ts["eps_perp"]/m["eps"], color=COLOR[r])
        ax[1, 1].plot(t, ts["eps_z"]/m["eps"], color=COLOR[r])
    m0 = next(iter(runs.values()))["meta"]
    ax[0, 1].axhline(1 - m0["sigma_eps"], color="0.45", lw=1, ls=":")
    ax[0, 1].annotate(r"$1-\sigma_\varepsilon$", (0.99, 1 - m0["sigma_eps"]), xycoords=("axes fraction", "data"),
                      ha="right", va="bottom", color="0.3")
    ax[0, 0].set(ylabel=r"$E/(\varepsilon\tau_A)$", yscale="log", title="total energy")
    ax[1, 0].set(ylabel=r"$E^\pm/(\varepsilon\tau_A)$", yscale="log", xlabel=r"$t/\tau_A$",
                 title=r"$E^+$ (solid) and $E^-$ (dashed)")
    ax[0, 1].set(ylabel=r"$\varepsilon_\perp/\varepsilon$", yscale="log", title="perpendicular dissipation")
    ax[1, 1].set(ylabel=r"$\varepsilon_z/\varepsilon$", yscale="log", xlabel=r"$t/\tau_A$", title="parallel dissipation")
    ax[0, 0].legend(loc="best")
    fig.tight_layout()

In [ ]:
# Budget and criteria 1-2, time-averaged over the second half of each run so far.
# dE/dt is the measured change of E over the window; inj_E is the REALISED injection rate
# (diagnostics.flrmhd.injection accumulated by the driver), not the requested eps.
rows = {}
for r, v in runs.items():
    ts, m = v["ts"], v["meta"]
    w = window(v); i0 = np.flatnonzero(w)[0]
    T = ts["t"][-1] - ts["t"][i0]
    dEdt = (ts["E"][-1] - ts["E"][i0])/T if T > 0 else np.nan
    dHdt = (ts["H"][-1] - ts["H"][i0])/T if T > 0 else np.nan
    q = {k: tmean(v, ts[k]) for k in ("inj_E", "inj_H", "eps_perp", "eps_z", "epsH_perp", "epsH_z")}
    # dissipation is sampled at block ends: average it with the trapezoid rule over the window
    trap = lambda y: np.sum(0.5*(y[i0+1:] + y[i0:-1])*np.diff(ts["t"][i0:]))/T if T > 0 else np.nan
    Dp, Dz = trap(ts["eps_perp"]), trap(ts["eps_z"])
    DH = trap(ts["epsH_perp"] + ts["epsH_z"])
    rows[r] = dict(eps=m["eps"], T=T/m["tau_A"], dEdt=dEdt, inj=q["inj_E"], Dp=Dp, Dz=Dz,
                   resid=dEdt - (q["inj_E"] - Dp - Dz), sig=q["inj_H"]/q["inj_E"],
                   residH=dHdt - (q["inj_H"] - DH), sigma_c=ts["sigma_c"][-1])
if rows:
    print(f"{'run':5s} {'window/tau_A':>12s} {'inj_E/eps':>10s} {'inj_H/inj_E':>12s} {'eps_perp/eps':>13s} "
          f"{'eps_z/eps':>10s} {'(dE/dt)/eps':>12s} {'E resid/eps':>12s} {'H resid/eps':>12s} {'sigma_c(end)':>13s}")
    for r, q in rows.items():
        e = q["eps"]
        print(f"{r:5s} {q['T']:12.2f} {q['inj']/e:10.4f} {q['sig']:12.4f} {q['Dp']/e:13.4f} {q['Dz']/e:10.4f} "
              f"{q['dEdt']/e:12.4f} {q['resid']/e:12.4f} {q['residH']/e:12.4f} {q['sigma_c']:13.3f}")
    print()
    note = "  [smoke profile: machinery check only, NOT a physics verdict]" if SMOKE else ""
    for r in ("bal", "rmhd"):
        if r in rows:
            q = rows[r]; tot = (q["Dp"] + q["Dz"])/q["eps"]
            print(f"F7.1 {r}: (eps_perp + eps_z)/eps = {tot:.3f} -> {'within' if abs(tot - 1) <= 0.15 else 'NOT within'} 15% of 1; "
                  f"eps_z/eps_perp = {q['Dz']/q['Dp']:.3g}{note}")
    if "imb" in rows:
        q = rows["imb"]; target = 1 - runs["imb"]["meta"]["sigma_eps_run"]
        print(f"F7.2 imb: (dE/dt)/eps = {q['dEdt']/q['eps']:+.3f} ({'growing' if q['dEdt'] > 0 else 'not growing'}); "
              f"budget residual {abs(q['resid'])/q['eps']:.3f} of eps ({'<=' if abs(q['resid']) <= 0.05*q['eps'] else '>'} 5%); "
              f"eps_perp/eps = {q['Dp']/q['eps']:.3f} against 1 - sigma_eps = {target:.2f} "
              f"(ratio {q['Dp']/q['eps']/target:.2f}; criterion: within a factor 2){note}")
        if "rmhd" in rows:
            print(f"F7.2 rmhd control: eps_perp/eps = {rows['rmhd']['Dp']/rows['rmhd']['eps']:.3f} (criterion: > 0.5){note}")

## Fluxes (criterion 3)

$\Pi(K)=-\sum_{k_\perp<K}T_E(k)$ and $\Pi_H(K)$ from the solver's own nonlinear term, averaged over the snapshots of the analysis window; $\Pi>0$ is transfer to small scales. A constant-flux range has $\Pi/\varepsilon\approx1$; behind the barrier the energy flux drops to $\sim(1-\sigma_\varepsilon)$ and the helicity flux to $\sim0$. Snapshot-cadence averages of an instantaneous flux are noisy: the shaded band is the standard error over snapshots.

In [ ]:
def averaged(v, key, frac=0.5):
    sp = spec_window(v, frac)
    a = np.array([s[key] for s in sp])
    return a.mean(axis=0), (a.std(axis=0)/np.sqrt(len(sp)) if len(sp) > 1 else np.zeros(a.shape[1:])), len(sp)

if any(v["spec"] for v in runs.values()):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True)
    for r, v in runs.items():
        if not v["spec"]:
            continue
        m = v["meta"]; K = v["spec"][-1]["K"]
        for a, key, norm in ((ax[0], "Pi", m["eps"]), (ax[1], "Pi_H", m["eps"])):
            y, e, n = averaged(v, key)
            a.plot(K, y/norm, color=COLOR[r], marker="o", ms=4, label=f"{hb.RUN_LABELS[r]} ({n} snapshots)")
            a.fill_between(K, (y - e)/norm, (y + e)/norm, color=COLOR[r], alpha=0.18, lw=0)
    for a in ax:
        a.axhline(0, color="0.4", lw=0.8); a.set(xscale="log", xlabel=r"$K/k_{\min}$")
        if m0["rho_i"] > 0:
            a.axvline(1/m0["rho_i"], color="0.45", lw=1, ls=":")
            a.annotate(r"$1/\rho_i$", (1/m0["rho_i"], 0.97), xycoords=("data", "axes fraction"), ha="right", va="top", color="0.3")
    ax[0].set(ylabel=r"$\Pi(K)/\varepsilon$", title="energy flux"); ax[1].set(ylabel=r"$\Pi_H(K)/\varepsilon$", title="helicity flux")
    ax[0].legend(loc="best", fontsize=8)
    fig.tight_layout()

## Spectra and the break (criterion 4)

$E^\pm(k_\perp)$ averaged over the analysis window. The break $k_*$ of $E^+$ is the continuous two-slope least-squares fit `diagnostics.flrmhd.spectral_break`, restricted to $k_\perp$ above the forcing shell ($k_\perp>2$) and below the dissipation range (here: $0.7\times$ the dealiased $k_{\perp,\max}$) — the estimator's sensitivity on real spectra is not established (docs), so the fit range is printed with the result.

In [ ]:
def fit_range(m):
    return 2.5, 0.7*m["n"]/3.0

def break_fit(k, E, m):
    # a two-slope fit has 4 parameters: ask for at least 6 bins in range, or it is not a fit
    kmin, kmax = fit_range(m)
    if np.count_nonzero((k >= kmin) & (k <= kmax) & (E > 0)) < 6:
        return None
    return dgf.spectral_break(k, E, kmin=kmin, kmax=kmax)

fits = {}
if any(v["spec"] for v in runs.values()):
    have = [r for r in runs if runs[r]["spec"]]
    fig, ax = plt.subplots(1, len(have), figsize=(4.2*len(have), 3.9), sharey=True, squeeze=False)
    for a, r in zip(ax[0], have):
        v = runs[r]; m = v["meta"]; k = v["spec"][-1]["k"]; u = m["eps"]*m["tau_A"]
        Ep, _, n = averaged(v, "E_plus"); Em, _, _ = averaged(v, "E_minus")
        pos = lambda y: np.where(y > 0, y, np.nan)        # empty bins are not drawn
        a.plot(k, pos(Ep)/u, color=COLOR[r], marker="o", ms=4, label=r"$E^+$")
        a.plot(k, pos(Em)/u, color=COLOR[r], ls="--", marker="s", ms=4, mfc="white", label=r"$E^-$")
        fits[r] = {s: break_fit(k, E, m) for s, E in (("+", Ep), ("-", Em))}
        f = fits[r]["+"]
        if f is not None:
            a.axvline(f["k_break"], color="0.3", lw=1)
            a.annotate(f"$k_*$ = {f['k_break']:.1f}\nslopes {f['slope_lo']:.2f}, {f['slope_hi']:.2f}",
                       (f["k_break"], 0.05), xycoords=("data", "axes fraction"), ha="left", va="bottom", fontsize=8)
        if m["rho_i_run"] > 0:
            a.axvline(1/m["rho_i_run"], color="0.45", lw=1, ls=":")
        a.set(xscale="log", yscale="log", xlabel=r"$k_\perp/k_{\min}$", title=f"{hb.RUN_LABELS[r]}\n({n} snapshots)")
        a.legend(loc="upper right")
    ax[0, 0].set_ylabel(r"$E^\pm(k_\perp)/(\varepsilon\tau_A)$")
    fig.tight_layout()
    for r in have:
        m = runs[r]["meta"]
        for s in "+-":
            f = fits[r][s]
            txt = ("no fit: fewer than 6 bins in the fit range" if f is None else
                   f"k* = {f['k_break']:.2f} (k* rho_i = {f['k_break']*m['rho_i']:.3f}), slopes {f['slope_lo']:.2f} | {f['slope_hi']:.2f}, rms resid {f['resid']:.2f}")
            print(f"{r:5s} E^{s}: fit range k in [{fit_range(m)[0]:.1f}, {fit_range(m)[1]:.1f}]: {txt}")
    if "imb" in fits and fits["imb"]["+"] is not None:
        f, m = fits["imb"]["+"], runs["imb"]["meta"]
        ok = f["k_break"]*m["rho_i"] < 1 and f["slope_hi"] < -3
        print(f"F7.4 imb: k* rho_i = {f['k_break']*m['rho_i']:.3f} (< 1?), slope beyond the break {f['slope_hi']:.2f} (steeper than -3?) -> "
              f"{'met' if ok else 'NOT met'}" + ("  [smoke profile: NOT a physics verdict]" if SMOKE else ""))

In [ ]:
# k* rho_i against (1 - sigma_c)^(1/4), one point per snapshot of the imbalanced run
if "imb" in runs and runs["imb"]["spec"]:
    v = runs["imb"]; m = v["meta"]
    pts = []
    for s in v["spec"]:
        f = break_fit(s["k"], s["E_plus"], m)
        if f is not None and float(s["sigma_c"]) < 1:
            pts.append((float(s["t"])/m["tau_A"], (1 - float(s["sigma_c"]))**0.25, f["k_break"]*m["rho_i"], f["slope_hi"]))
    if pts:
        t, x, y, sl = np.array(pts).T
        fig, ax = plt.subplots(figsize=(5.2, 4))
        sc = ax.scatter(x, y, c=t, cmap="Blues", vmin=0, edgecolor="0.3", lw=0.5, s=36, zorder=3)
        xx = np.linspace(x.min()*0.9, x.max()*1.1, 50)
        ax.plot(xx, xx, color="0.3", lw=1, label=r"$k_*\rho_i=(1-\sigma_c)^{1/4}$")
        ax.fill_between(xx, xx/2, xx*2, color="0.5", alpha=0.12, lw=0, label="factor 2")
        ax.set(xlabel=r"$(1-\sigma_c)^{1/4}$", ylabel=r"$k_*\rho_i$", title="break scale, imbalanced FLR-MHD")
        fig.colorbar(sc, label=r"$t/\tau_A$"); ax.legend(loc="best", fontsize=8)
        fig.tight_layout()
        inside = np.mean((y >= x/2) & (y <= 2*x))
        print(f"{len(pts)} snapshots with a fit; fraction within a factor 2 of (1 - sigma_c)^(1/4): {inside:.2f}"
              + ("  [smoke profile: NOT a physics verdict]" if SMOKE else ""))
    else:
        print("no snapshot of the imbalanced run has enough bins in the fit range for a break fit (expected on the smoke grid)")

## Where the energy is dissipated: the $(k_\perp,k_z)$ dissipation spectrum

$D(k_z,k_\perp)$ of the latest snapshot (the paper's Fig. 3c), normalized by its sum (linear axes, so the $k_z=0$ row is shown). In the saturated barrier state the dissipation moves from the perpendicular hyperviscous range (large $k_\perp$) to large $k_z$ at $k_\perp\rho_i<1$.

In [ ]:
have = [r for r in runs if runs[r]["spec"]]
if have:
    fig, ax = plt.subplots(1, len(have), figsize=(4.2*len(have), 3.8), sharey=True, squeeze=False)
    for a, r in zip(ax[0], have):
        s = runs[r]["spec"][-1]; m = runs[r]["meta"]
        D = s["D_kz_k"]/s["D_kz_k"].sum()
        kz = s["kz"]; k = s["k"]
        pc = a.pcolormesh(k, kz, np.log10(np.maximum(D, 1e-12)), cmap="Blues", vmin=-6, vmax=0, shading="nearest")
        a.grid(False)
        a.set_xlabel(r"$k_\perp/k_{\min}$")
        a.set_title(f"{hb.RUN_LABELS[r]}\n$t$ = {float(s['t'])/m['tau_A']:.2f} $\\tau_A$, {D[0].sum():.2f} at $k_z=0$", fontsize=9)
    ax[0, 0].set_ylabel(r"$k_z/k_{\min}$")
    fig.colorbar(pc, ax=ax[0].tolist(), label=r"$\log_{10}\,D(k_z,k_\perp)/\sum D$")

## Sub-filter transfers at $k_c<1/\rho_i$ (truth data for a closure)

$T^{\rm sf}_X(k)=T_X[f](k)-T_X[f^<](k)$ for $k_\perp<k_c$: the rate at which interactions with modes beyond the filter change the retained band's energy and helicity (plan §11). Its sum over $k$ is $-\Pi_X(k_c)$. A helicity-barrier closure must remove energy at $\sim(1-\sigma_\varepsilon)\varepsilon$ with $\sim$ zero net helicity removal.

In [ ]:
if have:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True)
    for r in have:
        v = runs[r]; m = v["meta"]; k = v["spec"][-1]["k"]; dk = k[1] - k[0]
        Tp, _, n = averaged(v, "T_sf_plus"); Tm, _, _ = averaged(v, "T_sf_minus"); TH, _, _ = averaged(v, "T_sf_H")
        low = k < float(v["spec"][-1]["kc"])
        ax[0].plot(k[low], (Tp + Tm)[low]*dk/m["eps"], color=COLOR[r], marker="o", ms=4, label=hb.RUN_LABELS[r])
        ax[1].plot(k[low], TH[low]*dk/m["eps"], color=COLOR[r], marker="o", ms=4)
        print(f"{r:5s} k_c = {float(v['spec'][-1]['kc']):g}: sub-filter energy flux Pi(k_c)/eps = {-(Tp + Tm).sum()*dk/m['eps']:+.3f}, "
              f"helicity flux Pi_H(k_c)/eps = {-TH.sum()*dk/m['eps']:+.3f}  ({n} snapshots)")
    for a in ax:
        a.axhline(0, color="0.4", lw=0.8); a.set(xlabel=r"$k_\perp/k_{\min}$")
    ax[0].set(ylabel=r"$T^{\rm sf}_E(k)\,\Delta k/\varepsilon$", title="sub-filter energy transfer")
    ax[1].set(ylabel=r"$T^{\rm sf}_H(k)\,\Delta k/\varepsilon$", title="sub-filter helicity transfer")
    ax[0].legend(loc="best", fontsize=8)
    fig.tight_layout()